In [1]:
import os
import GEOparse
import pandas as pd

# Get GEO_ID from parent folder name
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))

gse = GEOparse.get_GEO(geo=GEO_ID, silent=True)

# map sample title to metadata
metadata = {}

for key, value in gse.gsms.items():
    sample_title = value.metadata['title'][0]

    metadata_object = {'sample_title': sample_title}
    for item in value.metadata['characteristics_ch1']:
        k, v = item.split(':', 1)
        metadata_object[k.strip()] = v.strip()

    metadata[key] = metadata_object


metadata_df = pd.DataFrame.from_dict(metadata, orient='index')
metadata_df.index.name = 'samples'
# metadata_df = metadata_df.reset_index()

print(metadata_df.shape)
metadata_df.head()


(85, 19)


,sample_title,stic,age,path.stage,surgical.outcome,diagnosis.of.record,diagnosis.after.path.re.review,platinumfreeinterval.mos..,platinumstatus,recurrence,pfs.mos..,os.mos..,vitalstatus,race,tcga_subtype_dif,tcga_subtype_imr,tcga_subtype_mes,tcga_subtype_pro,tissue
samples,,,,,,,,,,,,,,,,,,,
GSM2722881,HGSC_40190,Yes,61,III,Optimal,Not available,Not available,Not available,Not available,Y,19,34,Dead,Not available,-0.156,0.609,-0.441,0.043,primary tumor tissue from high grade serous ov...
GSM2722883,HGSC_40191,No,49,III,Optimal,Not available,Not available,Not available,Not available,Y,40,64,Alive,Not available,0.443,0.487,-0.515,-0.311,primary tumor tissue from high grade serous ov...
GSM2722885,HGSC_40193,No,53,II,Unknown,Not available,Not available,Not available,Not available,Y,42,72,Alive,Not available,0.086,0.489,-0.27,-0.263,primary tumor tissue from high grade serous ov...
GSM2722888,HGSC_40197,Yes,53,III,Optimal,Not available,Not available,Not available,Not available,Y,32,34,Dead,Not available,0.348,0.383,-0.54,-0.085,primary tumor tissue from high grade serous ov...
GSM2722890,HGSC_40199,No,49,III,Suboptimal,Not available,Not available,Not available,Not available,Y,13,39,Dead,Not available,0.322,0.123,-0.491,0.147,primary tumor tissue from high grade serous ov...


# Use NCBI provided expression data

In [2]:
# Raw counts
ncbi_data_url = f"https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc={GEO_ID}&format=file&file={GEO_ID}_raw_counts_GRCh38.p13_NCBI.tsv.gz"


df_raw_counts: pd.DataFrame = pd.read_csv(ncbi_data_url, sep='\t', index_col=0, compression='gzip')
df_raw_counts = df_raw_counts.T
df_raw_counts.index.name = "samples"
df_raw_counts.columns.name = None

url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)
gene_mapping = (
    df_gene_annotations[["GeneID", "Symbol"]]
    .dropna(subset=["GeneID", "Symbol"])
    .set_index("GeneID")["Symbol"]
    .to_dict()
)
# Keep only columns that have a mapping
mappable = [c for c in df_raw_counts.columns if c in gene_mapping]
df_raw_counts = df_raw_counts[mappable].rename(columns=gene_mapping)
# df_raw_counts = df_raw_counts.rename(columns=gene_mapping)


df_combined = metadata_df.join(df_raw_counts, how='inner')
df_combined.reset_index()


data_file_name = f'{GEO_ID}_raw_counts_NCBI.csv'
df_combined.reset_index()
df_combined.to_csv(data_file_name, index=False)

df_combined

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_91288/3050319327.py:11: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(


,sample_title,stic,age,path.stage,surgical.outcome,diagnosis.of.record,diagnosis.after.path.re.review,platinumfreeinterval.mos..,platinumstatus,recurrence,...,ND4,TRNH,TRNS2,TRNL2,ND5,ND6,TRNE,CYTB,TRNT,TRNP
samples,,,,,,,,,,,,,,,,,,,,,
GSM2722881,HGSC_40190,Yes,61,III,Optimal,Not available,Not available,Not available,Not available,Y,...,18031,769,940,1259,23940,11935,1042,13270,1178,4209
GSM2722883,HGSC_40191,No,49,III,Optimal,Not available,Not available,Not available,Not available,Y,...,27333,765,1131,1468,29034,13910,1346,15287,908,4686
GSM2722885,HGSC_40193,No,53,II,Unknown,Not available,Not available,Not available,Not available,Y,...,35979,1659,2235,3048,47201,26147,2075,23360,1455,2888
GSM2722888,HGSC_40197,Yes,53,III,Optimal,Not available,Not available,Not available,Not available,Y,...,46103,1961,2446,3365,58297,29193,2473,32210,2412,4703
GSM2722890,HGSC_40199,No,49,III,Suboptimal,Not available,Not available,Not available,Not available,Y,...,24187,874,1124,1408,28622,12756,1354,17814,840,2010
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GSM2723048,HGSC_JH1072,Yes,54,IV,Optimal,Ovarian Ca,Ovarian Ca,41.4,Sensitive,N,...,21142,59,58,39,18902,3661,2721,14634,380,6111
GSM2723050,HGSC_JH1075,Yes,43,IV,Suboptimal,Ovarian Ca,Ovarian Ca,17.9,Sensitive,Y,...,16620,77,65,76,13222,1381,561,14345,145,1706
GSM2723052,HGSC_JH1085,No,63,IA,Optimal,Ovarian Ca,Ovarian Ca,50.3,Sensitive,N,...,30644,105,113,67,22131,2651,1342,22325,371,5296


In [3]:
import json
import os
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent  # adjust based on notebook depth
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from geo.scripts.generate_descriptions import generate_description_from_geo

description_file_path = 'description.json'


if not os.path.exists(description_file_path):
    with open(description_file_path, 'w') as f:
        json.dump({}, f)

with open(description_file_path, 'r') as f:
    desc_data = json.load(f)

if not desc_data:
    desc_data = generate_description_from_geo(GEO_ID)

desc_data['data_summary'] = {
    'samples': df_combined.shape[0],
    'clinical_features': metadata_df.shape[1],
    'genes': df_raw_counts.shape[1]
}
print("Description updated")

desc_data['data_file_names'] = [data_file_name]

with open(description_file_path, "w") as f:
    json.dump(desc_data, f, indent=4)

Description updated
